# Stage 2: Baseline Tokenization

## Step 1. Data Setup and Evaluation Protocol

This step defines the evaluation protocol: dataset split, sampled subset, baseline tokenizers, and target biomedical terms.


In [20]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from datasets import load_dataset
from IPython.display import display
from transformers import BertTokenizer, GPT2Tokenizer

RANDOM_SEED = 42
SAMPLE_SIZE = 300

TOKENIZER_SPECS = {
    "bert_base": "bert-base-uncased",
    "biobert": "dmis-lab/biobert-base-cased-v1.1",
    "gpt2": "gpt2",
}

TARGET_TERMS = [
    "nephrolithiasis",
    "gastroesophageal",
    "dermographism",
    "electrocardiogram",
    "immunohistochemistry",
    "acetylcholinesterase",
    "hypercholesterolemia",
    "hepatocellular",
    "neurodegenerative",
    "myocardial",
]


def get_context_text(context_field):
    if isinstance(context_field, dict):
        contexts = context_field.get("contexts", [])
        if isinstance(contexts, list):
            return " ".join(str(part) for part in contexts)
    if isinstance(context_field, list):
        return " ".join(str(part) for part in context_field)
    return str(context_field)


def count_words(text):
    return len(str(text).split())


def get_unk_token_candidates(tokenizer):
    candidates = set()
    if tokenizer.unk_token is not None:
        candidates.add(tokenizer.unk_token)
    if tokenizer.unk_token_id is not None:
        unk_from_id = tokenizer.convert_ids_to_tokens([tokenizer.unk_token_id])[0]
        candidates.add(unk_from_id)
    return candidates


def clean_token(token):
    return re.sub(r"^[\W_]+|[\W_]+$", "", token)


def evaluate_tokenizer(tokenizer, texts):
    unk_candidates = get_unk_token_candidates(tokenizer)
    token_counts = []
    word_counts = []
    unk_count = 0
    total_tokens = 0
    char_lengths = []
    vocab_counter = Counter()

    for text in texts:
        words = count_words(text)
        encoding = tokenizer(text, add_special_tokens=True)
        ids = encoding["input_ids"]
        tokens = tokenizer.convert_ids_to_tokens(ids)

        token_counts.append(len(tokens))
        word_counts.append(words)

        for token in tokens:
            total_tokens += 1
            vocab_counter[token] += 1
            if token in unk_candidates:
                unk_count += 1
            if token not in tokenizer.all_special_tokens:
                normalized = clean_token(token)
                if normalized:
                    char_lengths.append(len(normalized))

    return {
        "avg_tokens_per_sample": float(np.mean(token_counts)),
        "fertility": float(np.sum(token_counts) / max(np.sum(word_counts), 1)),
        "unk_rate": float(unk_count / max(total_tokens, 1)),
        "avg_token_length_chars": float(np.mean(char_lengths)) if char_lengths else 0.0,
        "top_tokens": vocab_counter.most_common(30),
    }


def token_split_examples(tokenizer, terms):
    rows = []
    for term in terms:
        ids = tokenizer(term, add_special_tokens=False)["input_ids"]
        pieces = tokenizer.convert_ids_to_tokens(ids)
        rows.append({"term": term, "pieces": pieces, "n_pieces": len(pieces)})
    return pd.DataFrame(rows)


In [21]:
dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled")["train"]

df = pd.DataFrame({
    "question": dataset["question"],
    "context_text": [get_context_text(x) for x in dataset["context"]],
    "label": dataset["final_decision"],
})

df["text"] = df["question"].fillna("") + " [SEP] " + df["context_text"].fillna("")
sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
sample_texts = sample_df["text"].tolist()

tokenizers = {
    "bert_base": BertTokenizer.from_pretrained(TOKENIZER_SPECS["bert_base"]),
    "biobert": BertTokenizer.from_pretrained(TOKENIZER_SPECS["biobert"]),
    "gpt2": GPT2Tokenizer.from_pretrained(TOKENIZER_SPECS["gpt2"]),
}

setup_summary_df = pd.DataFrame([
    {"metric": "labeled_dataset_size", "value": len(df)},
    {"metric": "evaluation_sample_size", "value": len(sample_df)},
    {"metric": "avg_words_per_sampled_text", "value": round(sample_df["text"].map(count_words).mean(), 2)},
    {"metric": "random_seed", "value": RANDOM_SEED},
])

tokenizer_setup_df = pd.DataFrame({
    "tokenizer_name": list(tokenizers.keys()),
    "model_id": [TOKENIZER_SPECS[k] for k in tokenizers.keys()],
    "family": ["WordPiece", "WordPiece", "BPE"],
})

display(setup_summary_df)
display(tokenizer_setup_df)


,metric,value
0,labeled_dataset_size,1000.00
1,evaluation_sample_size,300.00
2,avg_words_per_sampled_text,210.02
3,random_seed,42.00


,tokenizer_name,model_id,family
0,bert_base,bert-base-uncased,WordPiece
1,biobert,dmis-lab/biobert-base-cased-v1.1,WordPiece
2,gpt2,gpt2,BPE


### Step 1 Conclusion

- The setup summary reports **1,000 labeled samples**, with a fixed **300-sample evaluation subset** and `random_seed=42`.
- The baseline includes three tokenizers from two families: **WordPiece** (`bert_base`, `biobert`) and **BPE** (`gpt2`).


## Step 2. Core Baseline Metrics on Real PubMedQA Text

This step computes aggregate metrics (fertility, average tokens, UNK rate, token length) and ranks tokenizers.


In [22]:
metrics_rows = []
top_tokens_json = {}

for name, tokenizer in tokenizers.items():
    metrics = evaluate_tokenizer(tokenizer, sample_texts)
    metrics_rows.append({
        "tokenizer_name": name,
        "model_id": TOKENIZER_SPECS[name],
        "avg_tokens_per_sample": metrics["avg_tokens_per_sample"],
        "fertility": metrics["fertility"],
        "unk_rate": metrics["unk_rate"],
        "avg_token_length_chars": metrics["avg_token_length_chars"],
    })
    top_tokens_json[name] = metrics["top_tokens"]

metrics_df = pd.DataFrame(metrics_rows).sort_values(by="fertility").reset_index(drop=True)

best_f = metrics_df.loc[metrics_df["fertility"].idxmin()]
worst_f = metrics_df.loc[metrics_df["fertility"].idxmax()]

display(metrics_df)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (613 > 512). Running this sequence through the model will result in indexing errors


,tokenizer_name,model_id,avg_tokens_per_sample,fertility,unk_rate,avg_token_length_chars
0,gpt2,gpt2,313.383333,1.492136,0.0,5.010264
1,bert_base,bert-base-uncased,323.886667,1.542146,0.0,4.287131
2,biobert,dmis-lab/biobert-base-cased-v1.1,339.490000,1.616439,0.0,4.052037


### Step 2 Conclusion

- By fertility, the best tokenizer is **gpt2 (1.4921)** and the weakest is **biobert (1.6164)**.
- The fertility gap is **0.1243**, meaning biobert generates ~8% more tokens per word than gpt2 on average.
- `biobert` has the highest fertility despite being domain-adapted — suggesting its WordPiece vocabulary is less aligned with PubMedQA phrasing than expected.


## Step 3. Biomedical Term Fragmentation Analysis

This step measures how strongly each tokenizer splits biomedical terms from a fixed domain term list.


In [23]:
split_rows = []
for name, tokenizer in tokenizers.items():
    term_table = token_split_examples(tokenizer, TARGET_TERMS)
    term_table["tokenizer_name"] = name
    split_rows.append(term_table)

examples_df = pd.concat(split_rows, ignore_index=True)

term_summary = (
    examples_df.groupby("tokenizer_name")["n_pieces"]
    .agg(avg_pieces="mean", max_pieces="max", min_pieces="min")
    .sort_values("avg_pieces")
    .reset_index()
)

best_term = term_summary.iloc[0]
worst_term = term_summary.iloc[-1]

display(examples_df[["tokenizer_name", "term", "n_pieces", "pieces"]])
display(term_summary)


,tokenizer_name,term,n_pieces,pieces
0,bert_base,nephrolithiasis,6,"[ne, ##ph, ##rol, ##ith, ##ias, ##is]"
1,bert_base,gastroesophageal,7,"[gas, ##tro, ##es, ##op, ##ha, ##ge, ##al]"
2,bert_base,dermographism,4,"[der, ##mo, ##graph, ##ism]"
3,bert_base,electrocardiogram,4,"[electro, ##card, ##io, ##gram]"
4,bert_base,immunohistochemistry,6,"[im, ##mun, ##oh, ##isto, ##chemist, ##ry]"
5,bert_base,acetylcholinesterase,6,"[ace, ##ty, ##lch, ##olin, ##ester, ##ase]"
6,bert_base,hypercholesterolemia,5,"[hyper, ##cho, ##les, ##terol, ##emia]"
7,bert_base,hepatocellular,4,"[he, ##pa, ##to, ##cellular]"
8,bert_base,neurodegenerative,5,"[ne, ##uro, ##de, ##gen, ##erative]"
9,bert_base,myocardial,3,"[my, ##oca, ##rdial]"


,tokenizer_name,avg_pieces,max_pieces,min_pieces
0,bert_base,5.0,7,3
1,gpt2,5.1,7,3
2,biobert,5.3,7,4


### Step 3 Conclusion

All three baseline tokenizers split biomedical terms into multiple subwords. On the 10-term list, average split size is **5.0** for `bert_base`, **5.3** for `biobert`, and **5.1** for `gpt2`.

So for this term set, none of the baseline tokenizers keeps biomedical terms compact; this is the main motivation for training a domain-specific tokenizer in Stage 3.


## Step 4. Rare Long-Term Stress Test

This step stresses tokenizers on rare long biomedical-like words extracted from PubMedQA contexts.


In [24]:
all_words = []
for txt in sample_df["context_text"].fillna(""):
    all_words.extend(re.findall(r"[A-Za-z][A-Za-z\-]+", txt.lower()))

freq = Counter(all_words)
rare_long_terms = [w for w in set(all_words) if len(w) >= 12 and freq[w] <= 2]
rare_long_terms = sorted(rare_long_terms)[:30]

stress_rows = []
for name, tokenizer in tokenizers.items():
    avg_pieces = np.mean([
        len(tokenizer.convert_ids_to_tokens(tokenizer(w, add_special_tokens=False)["input_ids"]))
        for w in rare_long_terms
    ]) if rare_long_terms else 0.0
    stress_rows.append({
        "tokenizer_name": name,
        "avg_pieces_on_rare_long_terms": float(avg_pieces),
        "n_test_terms": len(rare_long_terms),
    })

stress_df = pd.DataFrame(stress_rows).sort_values("avg_pieces_on_rare_long_terms").reset_index(drop=True)

best_st = stress_df.iloc[0]
worst_st = stress_df.iloc[-1]

rare_long_terms_preview_df = pd.DataFrame({"rare_long_term": rare_long_terms[:15]})

display(rare_long_terms_preview_df)
display(stress_df)


,rare_long_term
0,absorptiometry
1,acceptability
2,accomplished
3,accumulation
4,acknowledged
5,adenocarcinomas
6,adenoidectomy
7,adenotonsillectomy
8,adipose-derived
9,administration-were


,tokenizer_name,avg_pieces_on_rare_long_terms,n_test_terms
0,bert_base,3.800000,30
1,biobert,4.066667,30
2,gpt2,4.100000,30


### Step 4 Conclusion

- Stress test uses **30** rare long terms.
- Best tokenizer on these terms: **bert_base (3.80 pieces/term)**.
- Weakest tokenizer on these terms: **gpt2 (4.10 pieces/term)**.
